# Milestone 1 — Colab CPU
Chạy các cell thiết lập theo thứ tự, rồi chọn **một** chế độ crawler. Sau khi kết nối Python mới, mount/chọn lại đúng Drive root, cập nhật code và nạp cấu hình trước khi xem status hoặc resume. Không cần dataset mới.

Bắt buộc có `SEMANTIC_SCHOLAR_API_KEY` trong Colab Secrets, bật Notebook access (hoặc biến môi trường). Không ghi key vào notebook. Không dùng Run all để khởi chạy các chế độ liên tiếp. Sau pull code mới, restart Python nếu crawler đã import, rồi chạy lại phần thiết lập.

## 1. Chọn Drive root và lấy/cập nhật code
Giữ quy trình mount Drive hiện có; chọn thư mục dự án đã mount ở cell dưới.

Bản worker/version: references truy vấn URL arXiv vN và lưu riêng từng version. Snapshot cũ v0 được giữ, không được coi là đã crawl cho v1…vN. Xem `docs/versioned-reference-workers.md`. Sau pull, nếu đã import crawler trong phiên Python này thì restart Python trước khi chạy tiếp; giữ nguyên root/range để resume dữ liệu cũ.


In [ ]:
from pathlib import Path
import os, subprocess, sys
selected = Path(input('Thư mục dự án đã chọn trên Google Drive: ')).expanduser()
assert selected.is_dir(), 'Chọn thư mục Drive đã mount và tồn tại'
assert input('Xác nhận thư mục này nằm trên Drive (yes): ') == 'yes'
os.chdir(selected)
print('Project root selected')

In [ ]:
repo = Path('code')
branch = 'feat/crawler-colab-optimization'
if not repo.exists():
    subprocess.run(['git','clone','-b',branch,'https://github.com/NewB-S2deeC0de/arXiv.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'fetch'],check=True)
    subprocess.run(['git','-C',str(repo),'checkout',branch],check=True)
    subprocess.run(['git','-C',str(repo),'pull'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(repo/'src/requirements.txt')],check=True)
sys.path.insert(0,str(repo/'src'))

## Khi nào bật TAKEOVER_STALE?
Mặc định False ở từng cell. Resume không yêu cầu bật cờ này. Nếu Stop kết thúc bình thường, giữ False.

Chỉ bật True khi gặp “Previous active marker exists” VÀ đã restart/terminate chính runtime cũ, đồng thời chắc chắn không còn notebook/runtime khác ghi cùng dataset. Mất mạng hoặc đóng tab không chứng minh Python cũ đã dừng. True cho phép tiếp quản dấu khóa, không dừng tiến trình khác, không reset cooldown hay failed jobs. Sau lần tiếp quản, đổi lại False.

Nếu cell không phản hồi Interrupt và phải restart, giữ đúng Drive root/range, chạy lại thiết lập và Status; tiếp quản nếu còn dấu khóa. Không xóa progress/journal để bỏ qua lỗi.

Khi bấm Stop, chương trình chờ các mục đã nhận vào queue và thread ghi hoàn tất rồi mới nhả lock. Chờ cell kết thúc; không bật takeover để mở thêm writer trong lúc đang drain. Nếu FUSE/Drive kẹt và buộc restart, chỉ tiếp quản sau khi chắc chắn runtime cũ đã bị dừng. Tiến độ được dựng lại từ dữ liệu bền vững, các response còn trên SSD có thể phải tải lại trong budget còn lại.


## 2. Nạp cấu hình và kiểm tra key
Nếu thiếu key, bổ sung rồi chạy lại cell này. Cell status vẫn có thể dùng sau lỗi thiếu key vì cấu hình đã nạp.


In [ ]:
from pathlib import Path
import subprocess, sys, traceback
from crawler.config import load_config
from crawler.pipeline import run_pipeline
from crawler.secrets import load_semantic_scholar_key, resolve_s2_key
from crawler.ids import generate_ids
from crawler.queue import run_range_queue, normalize_target, is_target_completed
from crawler.console import log

config = load_config(Path('code/config/crawler.yaml'))
config.execution['allow_anonymous_s2'] = False

# ==============================================================================
# DANH SÁCH RANGE CỦA CÁC THÀNH VIÊN TRONG NHÓM
# ==============================================================================
# - Mỗi range sẽ được tự động lưu vào 1 folder riêng biệt trên Google Drive:
#   datasets/{start_month}_{start_id}_to_{end_month}_{end_id}/
# - monthly_end_ids:
#     + Cùng tháng: để None (hoặc {}) như mẫu dưới.
#     + Xuyên tháng: nhập dict mapping tháng với ID cuối của tháng đó,
#       ví dụ: {'2025-04': 25480} nếu cào từ 2025-04 sang 2025-05.
# - s2_api_key:
#     + Để None: dùng SEMANTIC_SCHOLAR_API_KEY mặc định (trong Colab Secrets).
#     + Hoặc nhập tên Secret riêng (ví dụ: 'S2_KEY_MEMBER_2') hoặc API key trực tiếp.
# ==============================================================================
RANGES = [
    # Range của bạn: 2504.02523 -> 2504.07522 (5.000 bài)
    {
        'start_month': '2025-04',
        'start_id': 2523,
        'end_month': '2025-04',
        'end_id': 7522,
        'monthly_end_ids': None,  # Cùng tháng: để None hoặc {}
        's2_api_key': None,       # Dùng SEMANTIC_SCHOLAR_API_KEY mặc định
    },
    # Thêm range của 3 thành viên còn lại vào đây (bỏ dấu # để bật):
    # {
    #     'start_month': '2025-04',
    #     'start_id': 7523,
    #     'end_month': '2025-04',
    #     'end_id': 12522,
    #     'monthly_end_ids': None,
    #     's2_api_key': 'S2_KEY_MEMBER_2',  # Tên Secret trong Colab Secrets hoặc key trực tiếp
    # },
    # {
    #     'start_month': '2025-04',
    #     'start_id': 12523,
    #     'end_month': '2025-04',
    #     'end_id': 17522,
    #     'monthly_end_ids': None,
    #     's2_api_key': 'S2_KEY_MEMBER_3',
    # },
    # {
    #     'start_month': '2025-04',
    #     'start_id': 17523,
    #     'end_month': '2025-04',
    #     'end_id': 22522,
    #     'monthly_end_ids': None,
    #     's2_api_key': 'S2_KEY_MEMBER_4',
    # },
]

print(f'Đã nạp {len(RANGES)} range(s) hợp lệ:')
for idx, target in enumerate(RANGES, start=1):
    norm = normalize_target(target)
    sm, si, em, ei = norm['start_month'], norm['start_id'], norm['end_month'], norm['end_id']
    key_tag = f" | S2 Key: {norm.get('s2_api_key') or target.get('s2_api_key') or 'mặc định'}"
    print(f"  [{idx}] {sm}.{si:05d} → {em}.{ei:05d} | {len(norm['expected_ids'])} bài{key_tag} => Thư mục: {norm['folder']}")

first_target = normalize_target(RANGES[0])
config.range = {
    'start_month': first_target['start_month'],
    'start_id': first_target['start_id'],
    'end_month': first_target['end_month'],
    'end_id': first_target['end_id'],
}
if first_target['monthly_end_ids']:
    config.range['monthly_end_ids'] = first_target['monthly_end_ids']
config.paths['persistent_root'] = first_target['folder']
config.validate()

def require_s2_key(key_or_name=None):
    key = resolve_s2_key(key_or_name)
    if not key:
        name = key_or_name or 'SEMANTIC_SCHOLAR_API_KEY'
        raise RuntimeError(f'Thiếu {name}. Thêm trong Colab Secrets (biểu tượng chìa khóa), bật Notebook access, rồi chạy lại cell này.')
    return key

async def run_selected(stages, limit, takeover_stale=False):
    log('Notebook: run_selected entered; checking required S2 key')
    target1_key = first_target.get('s2_api_key') or RANGES[0].get('s2_api_key')
    key = require_s2_key(target1_key)
    log('Notebook: S2 key present; entering pipeline')
    config.execution['allow_anonymous_s2'] = False
    try:
        result = await run_pipeline(config, stages, resume=True, limit=limit,
                                    takeover_stale=takeover_stale, api_key=key)
    except BaseException:
        traceback.print_exc()
        raise
    if isinstance(result, dict) and 'api_wait_seconds' in result:
        api_wait = result['api_wait_seconds']
        wall = result.get('wall_seconds', 0.0)
        total_rl = api_wait.get('total_rate_limit_seconds', 0.0)
        ax_rl = api_wait.get('arxiv', {}).get('rate_limit_seconds', 0.0)
        s2_rl = api_wait.get('s2', {}).get('rate_limit_seconds', 0.0)
        effective = api_wait.get('effective_crawler_seconds', max(0.0, wall - total_rl))
        print(f"\n[Benchmark] Wall: {wall:.1f}s | Rate-limit stall: {total_rl:.1f}s (arXiv: {ax_rl:.1f}s, S2: {s2_rl:.1f}s) | Effective crawl: {effective:.1f}s")
    print(result)
    return result

print('\nPersistent root mặc định (Target 1):', config.paths['persistent_root'])
print('Commit:', subprocess.check_output(['git', '-C', 'code', 'log', '-1', '--oneline'], text=True).strip())
target1_key_name = first_target.get('s2_api_key') or RANGES[0].get('s2_api_key')
require_s2_key(target1_key_name)
print(f'S2 key cho Target 1 ({target1_key_name or "mặc định"}): đã tìm thấy (chưa kiểm chứng quyền truy cập API)')
print('S2 gap (giây):', config.network['s2_key_gap_seconds'])
print('S2 batch size:', config.network['s2_batch_size'])
print('Publish queue S2:', config.execution['s2_publish_queue_capacity'], 'batch/page /', config.execution['s2_publish_queue_bytes'], 'bytes')


## Scratch SSD tùy chọn
Chạy sau cell cấu hình để tải/giải nén trên SSD; result và progress vẫn lưu tại persistent_root trên Drive. Symlink hỏng sau runtime reset được tạo lại. Để False nếu muốn giữ scratch_root trong YAML.

Nên giữ `USE_LOCAL_SCRATCH=True` khi thử kiến trúc mới: raw pages S2 đang chờ publish nằm trên SSD; page cache đã publish nằm trên Drive. Queue giới hạn 8 page / 32 MiB kể cả mục đang ghi, ngoài queue tối đa một response 8 MiB đang tải/chờ. Đây là giới hạn payload, không phải toàn bộ RAM của Python.


In [ ]:
USE_LOCAL_SCRATCH = True
if USE_LOCAL_SCRATCH:
    import tempfile
    scratch_dir = Path('scratch')
    if scratch_dir.is_symlink() and not scratch_dir.exists():
        scratch_dir.unlink()
    if not scratch_dir.exists():
        scratch_dir.symlink_to(tempfile.mkdtemp(prefix='arxiv-crawler-'), target_is_directory=True)
    assert scratch_dir.is_dir()
    config.paths['scratch_root'] = 'scratch'
    config.validate()
print('Scratch project path:', config.paths['scratch_root'])


## Đồng bộ / Sửa journal_sequence (khi bị ngắt phiên đột ngột)
Chạy cell này nếu sau khi bị ngắt phiên (timeout 12h, mất mạng, restart) mà cell Status bên dưới báo lỗi `RuntimeError: Progress journal missing committed records required by snapshot`.
Cell này sẽ tự động kiểm tra số sequence hợp lệ thực tế trong `jobs.journal.jsonl` và đồng bộ lại `journal_sequence` trong `jobs.csv.json` mà không làm mất bất kỳ dữ liệu nào đã lưu.


In [ ]:
from crawler.journal import repair_journal_sequence

root = Path(config.paths["persistent_root"])
repair_result = repair_journal_sequence(root)
if repair_result["repaired"]:
    print(f"[Đã sửa] Đã đồng bộ journal_sequence về {repair_result['target_sequence']}.")
    print("Chi tiết:", repair_result["details"])
    print("Bây giờ bạn có thể chạy cell 3 (Status) bình thường.")
else:
    print(f"[Đồng bộ] Journal và snapshot khớp nhau (sequence: {repair_result['target_sequence']}). Không cần sửa.")


## 3. Status — xem sau khi ngắt hoặc kết nối lại
Không gọi mạng API, không reset job hay cooldown. Chạy sau thiết lập; nên chờ cell crawler dừng hẳn để đọc snapshot ổn định. Nếu báo progress hỏng/không đồng bộ, giữ nguyên dữ liệu để kiểm tra, không xóa journal.

Sau lần khởi chạy bản mới, `references` đếm số **version snapshots**; `legacy_references` đếm snapshot v0 cũ. 10 paper có thể có 14 version, vì thế không so hai số này như cùng đơn vị. `versions.csv` có trạng thái references mới; các cột references cũ ở `papers.csv` là legacy. S2 chạy batch qua pass 1–4 rồi pagination riêng ở pass 5; status hiển thị pass/cap đang chờ.


In [ ]:
from crawler.progress import ProgressStore
from crawler.queue import normalize_target
from datetime import datetime, timezone
import json, time
from collections import Counter

print(f'=== BÁO CÁO TIẾN ĐỘ CHO {len(RANGES)} RANGE TRONG QUEUE ===')
for idx, target in enumerate(RANGES, start=1):
    norm = normalize_target(target)
    root = Path(norm['folder'])
    progress = root / 'progress'
    marker = root / 'completed.json'
    sm, si, em, ei = norm['start_month'], norm['start_id'], norm['end_month'], norm['end_id']
    expected_count = len(norm['expected_ids'])
    print(f"\n{'='*70}")
    print(f"[{idx}/{len(RANGES)}] Range: {sm}.{si:05d} → {em}.{ei:05d} | {expected_count} bài ({norm['folder']})")
    print(f"{'='*70}")

    if marker.exists():
        try:
            info = json.loads(marker.read_text())
            print(f"  ● Trạng thái marker: ĐÃ HOÀN TẤT (completed.json lúc {info.get('completed_at')})")
        except Exception:
            print('  ● Trạng thái marker: ĐÃ HOÀN TẤT (completed.json)')
    else:
        print('  ● Trạng thái marker: Đang cào dở hoặc chưa chốt hoàn tất (chưa có completed.json)')

    if not (progress / 'jobs.csv').exists() and not (progress / 'jobs.journal.jsonl').exists():
        print('  ● Dữ liệu: [CHƯA BẮT ĐẦU] Thư mục chưa có dữ liệu tiến độ.')
        continue

    try:
        store = ProgressStore(root)
        counts = store.status_counts()

        print('\n  BẢNG THỐNG KÊ CHI TIẾT CÁC GIAI ĐOẠN (STAGES):')
        print(f"  {'Giai đoạn':<18} | {'Thành công':<12} | {'Failed (Lỗi)':<14} | {'No TeX / 404':<16} | {'Đang chờ':<10} | {'Tổng job':<10}")
        print(f"  {'-'*18}-+-{'-'*12}-+-{'-'*14}-+-{'-'*16}-+-{'-'*10}-+-{'-'*10}")

        total_failed_all = 0
        total_pending_all = 0
        for stage_name in ('discovery', 'metadata', 'source', 'references'):
            st_data = counts.get(stage_name, {})
            succ = st_data.get('succeeded', 0)
            fail = st_data.get('failed', 0)
            no_tex = st_data.get('no_tex', 0)
            not_found = st_data.get('not_found', 0)
            no_tex_or_nf = no_tex + not_found
            pend = st_data.get('pending', 0) + st_data.get('running', 0)
            st_total = sum(st_data.values())

            total_failed_all += fail
            total_pending_all += pend

            notes = []
            if no_tex > 0: notes.append(f"{no_tex} no_tex")
            if not_found > 0: notes.append(f"{not_found} 404")
            notes_str = f"{no_tex_or_nf:<4} ({', '.join(notes)})" if notes else f"{no_tex_or_nf:<16}"

            fail_str = f"❌ {fail:<11}" if fail > 0 else f"0{'':<13}"

            stage_label = {
                'discovery': '1. Discovery',
                'metadata': '2. arXiv Metadata',
                'source': '3. arXiv Source',
                'references': '4. S2 References',
            }.get(stage_name, stage_name)

            print(f"  {stage_label:<18} | {succ:<12} | {fail_str} | {notes_str} | {pend:<10} | {st_total:<10}")

        print(f"  {'-'*18}-+-{'-'*12}-+-{'-'*14}-+-{'-'*16}-+-{'-'*10}-+-{'-'*10}")
        print(f"  Tổng số job: {len(store.rows)} | Đang chạy/chờ (pending): {total_pending_all} | Bị lỗi (failed): {total_failed_all}")

        if total_failed_all > 0:
            error_reasons = Counter()
            sample_failed = []
            for k, r in store.rows.items():
                if r['status'] == 'failed':
                    err = r.get('error') or 'unknown'
                    error_reasons[err] += 1
                    if len(sample_failed) < 6:
                        sample_failed.append(f"{k[0]} (v{k[1]} {k[2]}: {err})")
            print(f"\n  ⚠️ CẢNH BÁO: Range này đang có {total_failed_all} job bị FAILED!")
            print(f"     Phân loại mã lỗi: {dict(error_reasons)}")
            print(f"     Mẫu một số ID bị lỗi: {sample_failed}")
            print("     -> Gợi ý: Hãy dùng Cell 6 (Retry Failed) bên dưới để cào bù tự động cho các bài này.")
        elif total_pending_all == 0:
            print("\n  ✅ HOÀN TẤT 100%: Toàn bộ dữ liệu của range đã cào trọn vẹn, không có job nào bị lỗi.")
        else:
            print(f"\n  ⏳ ĐANG TIẾN HÀNH: Còn {total_pending_all} job đang trong hàng đợi xử lý.")

    except Exception as exc:
        print(f'  Lỗi khi đọc progress: {exc}')

print('\nStatus chỉ đọc dữ liệu đã lưu; Drive có thể đồng bộ chậm.')


## 4. Smoke — tối đa 10 ID đầu
Đổi RUN_SMOKE=True rồi chạy cell. Resume dữ liệu của 10 ID đầu trong range; dữ liệu thành công được bỏ qua nên chạy lại có thể không gọi API. Không tự thử lại failed jobs.

Giới hạn là 10 **paper ID đầu**, gồm tất cả version đã discovery, không phải 10 request. Bản mới có thể phải crawl references vN dù snapshot v0 đã thành công. Không cam kết thời gian cố định: pagination, 429, Retry-After và Drive đều ảnh hưởng. Đợi kết thúc rồi chạy Status và kiểm tra mẫu `data/<paper>/versions/vN/references/normalized.json` (requested_identifier, requested_version, version_correspondence).


In [ ]:
RUN_SMOKE = False
TAKEOVER_STALE = False  # Chỉ True khi chắc chắn tiến trình cũ đã dừng.
if RUN_SMOKE:
    summary = await run_selected({'arxiv', 's2'}, limit=10, takeover_stale=TAKEOVER_STALE)
else:
    print('Chưa chạy smoke: đặt RUN_SMOKE=True khi sẵn sàng.')


## 5. Toàn bộ Queue Range — Chạy tuần tự tự động
Chạy tự động toàn bộ danh sách `RANGES` trong queue theo thứ tự:
1. Nếu range đã cào xong 100% (có file `completed.json`), crawler sẽ **tự động bỏ qua (skip)** trong tích tắc.
2. Nếu range đang cào dở hoặc chưa cào, hệ thống sẽ thực hiện Discovery rồi cào song song arXiv + S2 cho đến khi hoàn thành trọn vẹn.
3. Khi xong 100% một range, crawler tự động lưu snapshot, giải phóng lock, **dọn dẹp SSD scratch**, rồi lập tức chuyển sang range tiếp theo.

Dữ liệu của mỗi range được lưu độc lập tại thư mục `datasets/{start_month}_{start_id}_to_{end_month}_{end_id}/` trên Google Drive.


In [ ]:
RUN_FULL_RANGE = False  # Đổi True để chủ động chạy chế độ này.
TAKEOVER_STALE = False  # Chỉ True khi chắc chắn tiến trình cũ đã dừng.
if RUN_FULL_RANGE:
    for idx, t in enumerate(RANGES, start=1):
        require_s2_key(t.get('s2_api_key'))
    print(f'=== Bắt đầu cào tuần tự {len(RANGES)} range trong Queue ===')
    summaries = await run_range_queue(
        config,
        RANGES,
        stages={'discovery', 'arxiv', 's2'},
        takeover_stale=TAKEOVER_STALE,
    )
    print('\n=== Tổng kết toàn bộ các range ===')
    for s in summaries:
        target = s['target']
        target_info = f"{target['start_month']}.{target['start_id']:05d}->{target['end_month']}.{target['end_id']:05d}"
        status_str = f"Target: {target_info} ({s['folder']}) | Trạng thái: {s['status']}"
        summary = s.get('summary')
        if isinstance(summary, dict) and 'api_wait_seconds' in summary:
            api_wait = summary['api_wait_seconds']
            wall = summary.get('wall_seconds', 0.0)
            total_rl = api_wait.get('total_rate_limit_seconds', 0.0)
            ax_rl = api_wait.get('arxiv', {}).get('rate_limit_seconds', 0.0)
            s2_rl = api_wait.get('s2', {}).get('rate_limit_seconds', 0.0)
            effective = api_wait.get('effective_crawler_seconds', max(0.0, wall - total_rl))
            status_str += f"\n  -> Wall: {wall:.1f}s | Rate-limit stall: {total_rl:.1f}s (arXiv: {ax_rl:.1f}s, S2: {s2_rl:.1f}s) | Effective crawl: {effective:.1f}s"
        print(status_str)
else:
    print('Bỏ qua: RUN_FULL_RANGE=False')


## 6. Cào bù các job bị lỗi (Retry Failed)
Chạy cell này khi xem Status thấy có bài bị `failed` (do mạng chập chờn, timeout, hoặc arXiv/S2 từ chối tạm thời).
- Cell sẽ tự động xóa `completed.json` cũ (nếu có), reset các job bị `failed` về `pending`, và chạy lại đúng các bài lỗi này.
- Nếu có bài lỗi ở Discovery, nó sẽ tự động cào Discovery trước rồi mới cào arXiv Source, Metadata và S2.
- **Cộng dồn thời gian:** Toàn bộ thời gian chạy bù và thời gian chờ API sẽ được **tự động cộng dồn** vào báo cáo benchmark của các lần chạy trước.
- **FORCE_MARK_COMPLETED:** Mặc định `False`. Khi cào bù xong, nếu tất cả bài lỗi đều đã thành công (`failed == 0`), cell sẽ **tự động tạo lại `completed.json`**. Chỉ đổi sang `True` khi vẫn còn sót vài bài cá biệt (ví dụ bài bị tác giả thu hồi 404 thật) và bạn muốn chấp nhận kết thúc range để sang bước Export.


In [ ]:
RETRY_FAILED = False         # Đổi True để kích hoạt cào bù
TARGET_RANGE_INDEX = 1       # 1, 2, 3, 4 hoặc 'all' để quét toàn bộ các range
FORCE_MARK_COMPLETED = False # True nếu muốn ép chốt completed.json kể cả khi còn bài lỗi
TAKEOVER_STALE = False       # Chỉ True khi chắc chắn không còn tiến trình cũ chạy

if RETRY_FAILED:
    from crawler.queue import retry_range_queue
    if TARGET_RANGE_INDEX == 'all':
        selected_targets = RANGES
    else:
        idx = int(TARGET_RANGE_INDEX) - 1
        assert 0 <= idx < len(RANGES), f'TARGET_RANGE_INDEX phải từ 1 đến {len(RANGES)} hoặc "all"'
        selected_targets = [RANGES[idx]]

    for t in selected_targets:
        require_s2_key(t.get('s2_api_key'))

    print(f'=== Bắt đầu Retry / Cào bù cho {len(selected_targets)} range(s) ===')
    retry_results = await retry_range_queue(
        config,
        selected_targets,
        force_mark_completed=FORCE_MARK_COMPLETED,
        takeover_stale=TAKEOVER_STALE,
    )

    print('\n=== TỔNG KẾT KẾT QUẢ CÀO BÙ ===')
    for res in retry_results:
        t = res['target']
        folder = res['folder']
        info = f"{t['start_month']}.{t['start_id']:05d}->{t['end_month']}.{t['end_id']:05d}"
        print(f"\nTarget: {info} ({folder})")
        print(f"  - Trạng thái: {res['status']} | Completed marker: {'ĐÃ TẠO' if res['completed'] else 'CHƯA TẠO'}")
        print(f"  - Số job lỗi trước khi retry: {res['failed_before']}")
        print(f"  - Số job lỗi còn lại sau retry: {res['failed_after']}")
        if res['failed_after'] > 0:
            sample = [f"{k[0]} (v{k[1]} {k[2]})" for k in res['remaining_failed_keys'][:10]]
            print(f"    Mẫu các ID còn lỗi (tối đa 10): {sample}")
            if not res['completed']:
                print("    -> Chú ý: Chưa tạo completed.json do còn bài lỗi.")
                print("    -> Nếu bạn muốn chấp nhận kết quả này và chốt hoàn tất để Export,")
                print("       hãy đổi FORCE_MARK_COMPLETED = True ở đầu cell rồi chạy lại (chỉ mất 0.1s).")
        summary = res.get('summary')
        if isinstance(summary, dict) and 'api_wait_seconds' in summary:
            api_wait = summary['api_wait_seconds']
            wall = summary.get('wall_seconds', 0.0)
            total_rl = api_wait.get('total_rate_limit_seconds', 0.0)
            ax_rl = api_wait.get('arxiv', {}).get('rate_limit_seconds', 0.0)
            s2_rl = api_wait.get('s2', {}).get('rate_limit_seconds', 0.0)
            effective = api_wait.get('effective_crawler_seconds', max(0.0, wall - total_rl))
            active_wall = summary.get('active_wall_seconds', wall)
            print(f"  - Phiên cào bù này: Wall: {wall:.1f}s | Rate-limit stall: {total_rl:.1f}s | Effective: {effective:.1f}s")
            print(f"  - TỔNG THỜI GIAN CỘNG DỒN TOÀN BỘ CÁC LẦN CHẠY: {active_wall:.1f}s (~{active_wall/3600:.2f} giờ)")
else:
    print('Bỏ qua: RETRY_FAILED=False')


## Chẩn đoán source (tùy chọn)
Chỉ xem chẩn đoán, không reset failed. Failed jobs không được tự thử lại khi resume. Xem hướng dẫn retry trong README trước khi reset.


In [ ]:
from crawler.diagnostics import diagnose_sources
import json
print(json.dumps(diagnose_sources(config), indent=2, ensure_ascii=False))


## Export snapshot (tùy chọn)
Chỉ thực hiện sau khi crawler đã dừng.

Export schema 3 thêm `version_references.parquet` (liên kết version–snapshot–ref) và `reference_snapshots.parquet` (ledger/provenance, phân biệt null với rỗng). `references.parquet` là union cấp paper gồm cả legacy, đã loại trùng; không dùng union để kết luận bibliography của version.


In [ ]:
EXPORT_SNAPSHOT = False
if EXPORT_SNAPSHOT:
    from crawler.export import export_snapshot
    import uuid
    print(export_snapshot(config, 'export-' + uuid.uuid4().hex))


### Checkpoint và hiệu năng
Progress gồm jobs.csv và jobs.journal.jsonl; không xóa journal hoặc dùng code cũ sau nâng cấp. Chỉ phục hồi job dang dở trong phạm vi được chọn khi resume; trong phiên chỉ xác minh result vừa publish. Batch metadata theo IDvN giữ ngày từng phiên bản. Disk metrics không quét baseline; chỉ đo đường dẫn được quan sát trong phiên, không phải tổng dataset. Sau cập nhật, restart kernel, dùng đúng root cũ và smoke/interrupt trước full range.


## Đổi tháng và khoảng ID
Sửa range trong config/crawler.yaml của repo (trên Drive là code/config/crawler.yaml), rồi chạy lại cell load_config. start_month/end_month có dạng YYYY-MM, hỗ trợ khác tháng khi thêm monthly_end_ids cho tháng đầu và từng tháng trung gian; start_id/end_id là số nguyên, hai đầu inclusive. Đây là tháng trong arXiv ID, không phải khoảng ngày/giờ xuất bản bất kỳ. Với dataset mới, đổi paths.persistent_root sang thư mục tương đối mới; giữ root cũ khi resume cùng dataset. Xem README để có ví dụ đầy đủ.


## Dừng trước khi cập nhật code
Stop một lần và chờ cell dừng hẳn. Chưa restart/delete runtime khi Drive chưa đồng bộ; giữ backup toàn runtime tải về máy trước. Xem docs/safe-stop-and-upgrade.md trong repo để có cell backup và kiểm tra snapshot/journal. Giữ nguyên range/root khi resume dataset hiện tại.


## Theo dõi cell chạy
Log có giờ UTC và xuất ngay trước các thao tác có thể chậm. Dòng cuối cho biết thao tác vừa bắt đầu, chưa chứng minh nó đã treo. Nếu không thấy `Notebook: run_selected entered`, kiểm tra cell đã thực sự chạy và notebook/module đã cập nhật. Khi báo lỗi, gửi log từ đầu đến dòng cuối; không gửi API key. Log hiển thị trên output notebook, không phải checkpoint bền vững.


## Đo tổng dung lượng (tùy chọn, có thể rất lâu trên Drive)
Chỉ chạy khi crawler/audit đã dừng và bạn thực sự cần tổng logical bytes. Cell này quét toàn cây, không có checkpoint tiếp tục; không cần chạy để resume crawler. Kết quả được in riêng, không trộn vào metrics của phiên crawl.


In [ ]:
MEASURE_TOTAL_DISK = False
TAKEOVER_STALE = False
if MEASURE_TOTAL_DISK:
    from crawler.disk_usage import DiskUsage
    from crawler.locking import DatasetLock
    import uuid
    with DatasetLock(config.paths['persistent_root'], 'disk-' + uuid.uuid4().hex, TAKEOVER_STALE):
        print('Bắt đầu quét toàn bộ dung lượng; Drive có thể rất chậm.', flush=True)
        total_bytes = DiskUsage(config.paths.values()).total
        print('Tổng logical bytes trong persistent/scratch roots:', total_bytes, flush=True)
